<table style="width:100%; border-bottom: 2px solid #ccc; margin-bottom: 20px;">
  <tr>
    <td style="vertical-align:middle;">
         <img src="../../resources/ADI-Logo-RGB-FullColor.png" alt="Company Logo" height="30">
    </td>
    <td style="text-align:right; vertical-align:middle;">
      <p style="margin: 0;">Phased Array Systems</p>
      <p style="font-size: 14px; margin: 0;">Iain Derrington – ADEF Group, ADI</p>
      <p style="font-size: 12px; color: #555;">Field Applications & Platform Engineer</p>
    </td>
  </tr>
</table>

# CN0566 (Phaser) Setup and Validation

This notebook provides the required **one-time setup and validation steps** for the CN0566 (Phaser) platform.

Its purpose is to ensure that your host PC and Phaser hardware are correctly configured and communicating before proceeding to the Beamforming and FMCW tutorial notebooks.

The steps in this notebook focus on:
- Verifying software dependencies and versions
- Confirming hardware connectivity and control
- Running basic sanity checks to validate the end-to-end system

Once completed successfully, you should be able to execute the Beamforming and FMCW tutorial notebooks **without modification**.

> ⚠️ **Important**  
> The Phaser kit supports multiple configurations. However the tutorial within these notebooks assume the **Host Configuration**.
> In this configuration the host PC communicates directly to the RPi and PlutoSDR:
> 
> <img src="resources/host-config.png" alt="Kuiper Imager" height="30" width="400">



## Objective

By the end of this notebook you will be able to:
- Verify your host PC, PlutoSDR, and Raspberry Pi are correctly configured
- Confirm communication with the CN0566 (Phaser) hardware
- Execute the Beamforming and FMCW tutorial notebooks without modification


---

## Host PC Setup (Windows)

It is assumed if you are reading this from Jupyter notebooks you have read through the read.me  and installed Python + dependancies and setup a virtual enviroment. The next step is to install the drivers to allow communication with the hardware.

### Drivers
The setup described here can also be found [here](https://analogdevicesinc.github.io/documentation/tools/pluto-m2k/drivers.html#pluto-m2k-drivers-windows).

* Download and install the Pluto Drivers from:  https://github.com/analogdevicesinc/plutosdr-m2k-drivers-win/releases
  * This driver includes WinUSB / driver bindings for the ADALM-PLUTO (PlutoSDR) and ADALM2000 (M2K)
* Dowlonad and install IIO from here: https://github.com/analogdevicesinc/libiio/releases
  *  The IIO drivers provide the software infrastructure to communicate over USB / Ethernet with many ADI parts including Phaser

---

## PlutoSDR Configuration

The Pluto that ships with the Phaser kit has been pre-configured and should not need to be modified.  However, in case something goes wrong or the firmware needs to be updated, this page details the Pluto setup procedures. 

### Updating Pluto Firmware
The latest Pluto firmware is found here:

[https://github.com/analogdevicesinc/plutosdr-fw/releases](https://github.com/analogdevicesinc/plutosdr-fw/releases)

**It is recommended that all Phaser's operate with version 0.39, or later**.  Details on how to perform the firmware update are found here:

[https://wiki.analog.com/university/tools/pluto/users/firmware](https://wiki.analog.com/university/tools/pluto/users/firmware)


### Pluto Configuration

The next step is to update the Pluto configuration to enable the AD9361's second channel. Follow the directions at: [Updating to the AD9364](https://wiki.analog.com/university/tools/pluto/users/customizing#updating_to_the_ad9364)

For setting the mode of a Rev. C PlutoSDR to 2r2t, login to Pluto (via PuTTY) and enter the following sequence of commands:

```{code-block} bash
fw_setenv attr_name compatible
fw_setenv attr_val ad9361
fw_setenv compatible ad9361
fw_setenv mode 2r2t
reboot
```


Next, verify that the configuration was programmed properly by entering the following commands:
```{code-block} bash
fw_printenv attr_name
fw_printenv attr_val
fw_printenv compatible
fw_printenv mode
```

Which should return:
```{code-block} bash
# fw_printenv attr_name
attr_name=compatible
# fw_printenv attr_val
attr_val=ad9361
# fw_printenv compatible
compatible=ad9361
# fw_printenv mode
mode=2r2t
#
```

For questions or help with the Phaser, please visit:
[https://ez.analog.com/adieducation/university-program/](https://ez.analog.com/adieducation/university-program/)

---

## Raspberry Pi Setup (Phaser Controller)

### Download and Image the SD Card
The ADALM-PHASER kit ships with an SD card; however, this SD card must be updated with a new image. The Phaser software is tested with Kuiper Linux 2021_R2, which, in spite of the “2021” in the name, was released in Spring, 2023. While there are newer Kuiper Linux releases, this version is required to maintain compatibility with the current phaser software packages.

This image can be downloaded here: [https://swdownloads.analog.com/cse/kuiper/image_2023-04-02-ADI-Kuiper-full.zip](https://swdownloads.analog.com/cse/kuiper/image_2023-04-02-ADI-Kuiper-full.zip)

Unzip the file and burn the image onto the SD card using the traditional methods. This typically means using Win32DiskImager (Windows) or Etcher (Linux).

If additional guidance is required, more information can be found here:

https://wiki.analog.com/resources/tools-software/linux-software/zynq_images/windows_hosts
https://wiki.analog.com/resources/tools-software/linux-software/zynq_images/linux_hosts

**For ADI employees, please use ADI-Kuiper Imager, as it works around the SD card encryption.**

<img src="resources/kuiper-imager.png" alt="Kuiper Imager" height="30" width="400">

After writing the image, if a window pops up saying “this card needs to be formatted, would you like to format it now?”, the answer is NO. Eject the card and insert it into the Raspberry Pi SD card slot.

### Configuring the Raspberry Pi

The goal of this step is to run a setup script that configures the Raspberry Pi for use with the Phaser. The script downloads and installs all required software and perform the system configuration.

The setup script is available here:
https://github.com/thorenscientific/rpi_setup_stuff/raw/main/phaser/phaser_sdcard_setup.sh

The simplest way to execute the setup script is to provide the Phaser with internet access. This can be achieved using either a wired Ethernet connection (`eth0`) or Wi-Fi (`wlan0`). 

**Internet access is only required for setup**. The tutorials in the other notebooks do not require the RPi to have internet! The Host Configuration requires that ('eth0') is untimately configured with a static IP address so it can be accessed direct from a PC, with no dependance on an external network.

Insert the newly programmed SD card, power the Phaser kit using the supplied USB-C power connector (use the power socket on the main Phaser board, not the Raspberry Pi USB-C connector). Wait for the Raspberry Pi to boot before proceeding.

In theory, a wired Ethernet connection to your local network should allow you to access the Raspberry Pi using VNC at `analog.local`. If this works, you can proceed directly to the setup script section.

If you encounter issues with a wired Ethernet connection, configure the Raspberry Pi locally by connecting a monitor, keyboard, and mouse to the Raspberry Pi.

### Enabling / Configuring WIFI

To enable WiFi directly on the Raspberry Pi, use the RPi GUI:

<img src="resources/RPiGUI.png" alt="Kuiper Imager" height="30" width="400">

* Click the WIFI icon on the top right of the screen and select country of residence
* Enter password 'analog'
* Select the country your in
* Click WiFi icon on the top right of the screen and select Wireless router and enter password to connect.


> ⚠️ **Important**  
> Confirm WIFi is working before proceeding. You will need access to the internet to download Setup Script

Open a termial, by clicking the terminal icon at the top right of the page and type:

```bash
ip a
```

Note the IP address for the *wlan0* connection. e.g. 192.168.0.86:

<img src="resources/ip-a.png" alt="Kuiper Imager" height="30" width="400">

Check you can [VNC](https://www.realvnc.com/en/connect/download/viewer/) or SSH (User name: root, Password: analog) into the Rasberry Pi from a PC. The device should be assinged the address analog.local but if that fails try using the address recorded earlier.

### Enabling / Configuring Ethernet

You may not have access to WiFi in which case the Ethernet ('eth0') port can be brought up by following these instructions

```bash
ip link show eth0
```
You should see ('eth0') listed

Bring up the Ethernet interface:

```bash
sudo ip link set eth0 up
```

Request an IP address via DHCP:

```bash
sudo udhcpc -i eth0
```

Expected output:  udhcpc: obtained lease

Once Ethernet is enabled, the Raspberry Pi will obtain an IP address via DHCP.
Access via analog.local is not guaranteed on Kuiper Linux; if mDNS is not available, connect using the assigned IP address instead. To get the IP address execute:

```bash
ip a
```

<details>
<summary><strong>More Information on mDNS</strong></summary>
If you want to check if Multicast DNS is running, execute:

```bash
systemctl status avahi-daemon
```

If its up and running you should see something along the lines of:

```
● avahi-daemon.service - Avahi mDNS/DNS-SD Stack
     Loaded: loaded (/lib/systemd/system/avahi-daemon.service; enabled; vendor preset: enabled)
     Active: active (running) since Tue 2026-01-13 16:54:50 GMT; 2min 57s ago
TriggeredBy: ● avahi-daemon.socket
   Main PID: 511 (avahi-daemon)
     Status: "avahi-daemon 0.8 starting up."
      Tasks: 2 (limit: 3694)
        CPU: 313ms
     CGroup: /system.slice/avahi-daemon.service
             ├─511 avahi-daemon: running [phaser.local]
             └─517 avahi-daemon: chroot helper

Jan 13 16:54:56 phaser avahi-daemon[511]: Joining mDNS multicast group on interface wlan0.IPv6 with address fe80::c9bc:8094:167d:2d3f.
Jan 13 16:54:56 phaser avahi-daemon[511]: New relevant interface wlan0.IPv6 for mDNS.
Jan 13 16:54:56 phaser avahi-daemon[511]: Registering new address record for fe80::c9bc:8094:167d:2d3f on wlan0.*.
Jan 13 16:54:56 phaser avahi-daemon[511]: Joining mDNS multicast group on interface eth0.IPv6 with address fe80::bb79:b606:1017:cc9c.
Jan 13 16:54:56 phaser avahi-daemon[511]: New relevant interface eth0.IPv6 for mDNS.
Jan 13 16:54:56 phaser avahi-daemon[511]: Registering new address record for fe80::bb79:b606:1017:cc9c on eth0.*.
Jan 13 16:55:01 phaser avahi-daemon[511]: Joining mDNS multicast group on interface wlan0.IPv4 with address 192.168.0.86.
Jan 13 16:55:01 phaser avahi-daemon[511]: New relevant interface wlan0.IPv4 for mDNS.
Jan 13 16:55:01 phaser avahi-daemon[511]: Registering new address record for 192.168.0.86 on wlan0.IPv4.
Jan 13 16:55:07 phaser avahi-daemon[511]: Registering new address record for 169.254.106.254 on eth0.IPv4.
```
</details> <br>

Use ping to verify Ethernet / Internet connectivity. The device should be ready to download and execute the setup script.

Once the Rasperry Pi is on the network you can use SSH or VNC from a PC if so desired. Use the address obtained

### Setup Script
If using RPi directly or through the VNC, switch to the root user, by typing the following in terminal:
```bash
sudo -i
```

You will be prompted for the password. Enter **analog**.  Confirm the prompt has changed to root@analog:~#

Execute the following commands on the Rasperry Pi, making sure to put in the write date and aproximately the right time.

```bash
sudo date -s "2026-MM-DD HH:MM:00"
sudo apt update
sudo apt install -y systemd-timesyncd ca-certificates
sudo timedatectl set-ntp true
```


<details>
<summary>Command details</summary>
This above commands installs two system-level components.

systemd-timesyncd
* A lightweight NTP client provided by systemd.
* It automatically synchronises the system clock with internet time servers.
* Designed for embedded / headless systems (like Pi / Kuiper).

ca-certificates
* Installs the trusted root Certificate Authorities used by Linux.
* These certificates are required to verify HTTPS/TLS connections.

sudo timedatectl set-ntp true

Enables time synchronisation at the system level.
</details> <br>

> ⚠️ **Important**  
> If you haven't set the correct time and date, it is very likley the following command will fail 

Execute the following commands to retrieve the phase_sdcard_setup script:

```bash
wget https://github.com/thorenscientific/rpi_setup_stuff/raw/main/phaser/phaser_sdcard_setup.sh 
sudo chmod +x phaser_sdcard_setup.sh
```

Finally run the setup script:

```bash
./phaser_sdcard_setup.sh
```

This will setup the RPi for use with the Phaser kit. For more information about what the script does expand the following section:

<details>
    <summary>Script Details</summary>
<section>
  <dl>
    <dt><strong>Bootstraps system time</strong></dt>
    <dd>
      <ul>
        <li>Sets the system clock using the HTTP <code>Date</code> header from Google to work around systems without an RTC.</li>
        <li>Updates the APT package cache.</li>
      </ul>
    </dd>
    <dt><strong>Configures Raspberry Pi boot settings for Phaser</strong></dt>
    <dd>
      <ul>
        <li>Replaces <code>/boot/config.txt</code> with a Phaser-specific configuration.</li>
        <li>Backs up the original boot config.</li>
        <li>Enables required device-tree overlays, LED heartbeat behaviour, and a hardware shutdown button.</li>
      </ul>
    </dd>
    <dt><strong>Enables automatic PlutoSDR sharing via iiod</strong></dt>
    <dd>
      <ul>
        <li>Installs a udev rule and a systemd template service.</li>
        <li>Automatically launches a second <code>iiod</code> instance when a PlutoSDR is connected.</li>
        <li>Supports hot-plug and reconnect without reboot.</li>
        <li>Exposes the Pluto over a dedicated TCP port (<code>12345</code>).</li>
      </ul>
    </dd>
    <dt><strong>Sets system hostname to <code>phaser</code></strong></dt>
    <dd>
      <ul>
        <li>Replaces <code>/etc/hostname</code> and <code>/etc/hosts</code> (with backups).</li>
        <li>Distinguishes Phaser systems from other ADI platforms using the default analog hostname.</li>
      </ul>
    </dd>
    <dt><strong>Installs a helper script for Pluto AD9361 mode</strong></dt>
    <dd>
      <ul>
        <li>Downloads and enables a convenience script to switch Pluto firmware configuration.</li>
      </ul>
    </dd>
    <dt><strong>Forces a known-good <code>pyadi-iio</code> version</strong></dt>
    <dd>
      <ul>
        <li>Removes any existing <code>pyadi-iio</code> installation.</li>
        <li>Builds and installs the current main branch directly from source.</li>
        <li>Ensures compatibility with Phaser support prior to packaged releases.</li>
      </ul>
    </dd>
    <dt><strong>Upgrades Python plotting dependencies</strong></dt>
    <dd>
      <ul>
        <li>Installs or upgrades <code>pyqtgraph</code> to support Phaser GUI-based scripts.</li>
      </ul>
    </dd>
    <dt><strong>Refreshes dynamic linker cache</strong></dt>
    <dd>
      <ul>
        <li>Runs <code>ldconfig</code> to ensure newly installed libraries are correctly resolved.</li>
      </ul>
    </dd>
    <dt><strong>Installs SSH automation utility</strong></dt>
    <dd>
      <ul>
        <li>Installs <code>sshpass</code> for scripted or automated SSH access to a connected Pluto.</li>
      </ul>
    </dd>
    <dt><strong>Pre-downloads PlutoSDR firmware</strong></dt>
    <dd>
      <ul>
        <li>Fetches Pluto firmware <code>v0.38</code> for later flashing or updates.</li>
      </ul>
    </dd>
    <dt><strong>Provides post-setup guidance</strong></dt>
    <dd>
      <ul>
        <li>Recommends a reboot.</li>
        <li>Suggests disabling VNC and switching to CLI-only boot for faster startup on headless systems.</li>
      </ul>
    </dd>
  </dl>
</section>
</details>


### Static IP Address (eth0)

Its often easier and simpler to assign a static IP address to the RPi so the host PC can communicate direclty with the RPi, without an external network.

Follow these instructions to assign a static IP address to ('Eth0')

Execute the following commands in a terminal on the RPi to create a new file:

```bash
sudo nano /etc/network/interfaces.d/eth0
```

Copy and paste the following in to the new file:
```bash
auto eth0
iface eth0 inet static
    address 192.168.1.10
    netmask 255.255.255.0
    gateway 192.168.1.1
```

Save the file and exit out of the nano editor.

Execute the following commands

```bash
sudo ifdown eth0 2>/dev/null
sudo ifup eth0
```

Reboot the RPi

```bash
reboot
```

Connect the ethernet port of the RPi to a local PC. You will need to modify the adapter settings on the PC. Assuming you are using a Windows machine

```windows
Win + R → ncpa.cpl → Enter
```

<img src="resources/windows-r.png" alt="Advanced Network Settings" height="30" width="400">

Select the correct NIC / adapter to which the RPi is connected. Right click and select properties

<img src="resources/adapters.png" alt="Advanced Network Settings" height="30" width="400">

Select IP4 settings and click properties:

<img src="resources/advanced-network-settings.png" alt="Advanced Network Settings" height="30" width="400">

Enter the PC IP address and Mask.

<img src="resources/ip4-settings.png" alt="Advanced Network Settings" height="30" width="400">

Once the RPi has rebooted, open an SSH session to the Rasperry Pi. You should be able to use the newly assigned static IP address.

---

## Verification and Sanity Checks



In [3]:

import sys
import numpy as np

from adi import ad9361
from adi.cn0566 import CN0566

labels = ["Board Temperature"]

# -----------------------------
# USER SETTINGS
# -----------------------------
CN0566_IP_ADDRS = "192.168.1.10"   # CN0566 uses Pluto's IIO context (same URI in most setups)
PLUTO_IP_ADDRS  = "192.168.2.1"

try:
    print(f"Attempting to connect to CN0566 via ip:{CN0566_IP_ADDRS}...")
    my_phaser = CN0566(uri="ip:" + CN0566_IP_ADDRS)
    print("CN0566 connected :)")

    print(f"Connecting to PlutoSDR via ip:{PLUTO_IP_ADDRS}...")
    my_sdr = ad9361(uri="ip:" + PLUTO_IP_ADDRS)
    print("PlutoSDR connected :)")

except Exception as e:
    print("Unable to connect to CN0566 or PlutoSDR. Please check IP addresses and connections.")
    print("Error:", e)
    sys.exit(1)

# Set my_phaser.sdr to instance of PlutoSDR
my_phaser.sdr = my_sdr
my_phaser.sdr._ctx.set_timeout(10000)  # 10 seconds

print("\nBasic checks:")

try:
    print("Reading Phaser monitor voltages (quick comms check)...")
    mon = my_phaser.read_monitor()
    # read_monitor returns a dict-like structure; print a couple of keys if present
    if mon != None:
        print(f"{labels[0]} = {mon[0]:.2f} C")
    else:
        print("Phaser monitor returned no data (still may be OK depending on fw).")
except Exception as e:
    print("Phaser monitor read failed (Phaser comms may not be correct).")
    print("Error:", e)
    sys.exit(1)

# -----------------------------
# 2) Pluto RX streaming sanity
# -----------------------------
try:
    print("Configuring Pluto RX (sanity defaults)...")
    my_sdr.sample_rate = int(2e6)
    my_sdr.rx_rf_bandwidth = int(2e6)
    my_sdr.rx_lo = int(2.45e9)
    my_sdr.gain_control_mode_chan0 = "slow_attack"
    my_sdr.rx_buffer_size = 16384

    print("Capturing RX samples...")
    x = my_sdr.rx()
    x = np.asarray(x)

    pwr = float(np.mean(np.abs(x)**2))
    peak = float(np.max(np.abs(x)))
    zeros = int(np.sum(x == 0))

    print(f"RX OK: N={len(x)}, mean_power={pwr:.3e}, peak={peak:.3f}, zeros={zeros}")

    if pwr == 0:
        print("WARNING: RX power is zero (could be fine if antenna/path quiet, but check RF chain).")
    if peak > 0.95:
        print("WARNING: Possible clipping (reduce gain or strong signal present).")

except Exception as e:
    print("Pluto RX capture failed (Pluto streaming may not be correct).")
    print("Error:", e)
    sys.exit(1)

print("\nComplete...")

Attempting to connect to CN0566 via ip:192.168.1.10...
CN0566 connected :)
Connecting to PlutoSDR via ip:192.168.2.1...
PlutoSDR connected :)

Basic checks:
Reading Phaser monitor voltages (quick comms check)...
Board Temperature = 38.00 C
Configuring Pluto RX (sanity defaults)...
Capturing RX samples...
RX OK: N=2, mean_power=3.821e+02, peak=73.246, zeros=17

Complete...
